In [ ]:
import pandas as pd
import gc
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    root_mean_squared_error,
    r2_score,
    mean_absolute_percentage_error,
    median_absolute_error,
    explained_variance_score,
)
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor

In [38]:
df = pd.read_parquet("data/flight.parquet", engine="pyarrow")
df

,flight,payload,route,altitude_preset,date,time_day,duration_s,total_distance_m,wind_speed_mean,wind_speed_std,wind_x_mean,wind_y_mean,speed_mean,speed_max,velocity_mag_mean,velocity_mag_max,acceleration_mag_mean,acceleration_mag_std,battery_voltage_mean,battery_voltage_min,max_height_agl,final_height_agl,energy_consumed_Wh,battery_needed_mAh,landing_offset_flag
0,1,0.0,R5,25,2019-04-07,10:13,200.70,549.065984,3.898058,1.952675,0.973186,-1.246141,4.0,4.0,2.676774,6.261616,9.842870,0.466372,22.070134,21.228519,26.257755,-0.994409,21.769975,1000.743935,False
1,2,0.0,R5,50,2019-04-07,10:23,271.20,666.615556,3.522941,2.159456,0.088843,-1.886357,4.0,4.0,2.387432,7.676739,9.881874,0.628406,21.527547,20.125463,52.637988,0.589477,25.366627,1205.255874,False
2,3,0.0,R5,25,2019-04-07,10:33,180.10,577.009957,4.581182,3.335733,-0.696895,-1.778235,6.0,6.0,3.110644,7.213987,9.902090,0.545290,22.330305,19.943916,24.660462,0.106140,17.094392,789.073853,False
3,4,0.0,R5,25,2019-04-07,10:48,171.00,562.802357,4.596319,3.438072,-0.351732,-1.389653,8.0,8.0,3.165734,9.425537,9.900368,0.559073,21.950616,20.365856,25.580572,0.416669,14.690038,687.813976,False
4,5,0.0,R2,25,2019-04-07,11:05,217.00,470.978276,3.333910,2.247522,0.062744,-1.257258,4.0,4.0,2.126439,4.900079,9.817243,0.341981,21.519937,18.923494,24.323036,-0.924901,19.019928,920.070980,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
204,275,500.0,R1,25,2019-10-24,9:05,149.40,518.231162,4.139878,3.885389,0.336196,-0.676045,8.0,8.0,3.336389,8.788151,9.850903,0.420902,22.056616,20.332052,21.959932,-3.264891,16.295811,760.480643,False
205,276,500.0,R1,25,2019-10-24,9:32,147.90,517.758034,4.392581,4.332293,0.252633,-0.635730,10.0,10.0,3.402998,10.553163,9.862172,0.471081,21.492353,19.788662,22.542598,1.293232,15.392088,738.620356,False
206,277,500.0,R1,25,2019-10-24,9:45,134.81,517.677109,5.524651,4.029744,-0.785881,-1.457703,10.0,10.0,3.680124,10.579715,9.847796,0.529103,21.908016,19.352947,24.289124,-0.286362,15.531389,741.451254,False
207,278,500.0,R7,25-50-100-25,2019-10-24,10:00,186.39,545.413261,4.686967,3.826570,-0.546397,-1.806563,10.0,10.0,2.907930,10.376503,9.829065,0.456918,22.394109,20.407175,95.041887,-0.457578,18.922540,887.198699,False


In [39]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 209 entries, 0 to 208
Data columns (total 25 columns):
 #   Column                 Non-Null Count  Dtype   
---  ------                 --------------  -----   
 0   flight                 209 non-null    int64   
 1   payload                209 non-null    float64 
 2   route                  209 non-null    category
 3   altitude_preset        209 non-null    string  
 4   date                   209 non-null    str     
 5   time_day               209 non-null    str     
 6   duration_s             209 non-null    float64 
 7   total_distance_m       209 non-null    float64 
 8   wind_speed_mean        209 non-null    float64 
 9   wind_speed_std         209 non-null    float64 
 10  wind_x_mean            209 non-null    float64 
 11  wind_y_mean            209 non-null    float64 
 12  speed_mean             209 non-null    float64 
 13  speed_max              209 non-null    float64 
 14  velocity_mag_mean      209 non-null    float64 
 15  

In [40]:
selection_cols = [
    'total_distance_m',
    'max_height_agl',
    'payload',
    'wind_speed_mean',
    'acceleration_mag_std',
    'velocity_mag_mean',
    'wind_speed_std',
    'wind_y_mean',
    'wind_x_mean',
    'acceleration_mag_mean',
    'energy_consumed_Wh'
]
target = 'energy_consumed_Wh'

In [41]:
X = df[selection_cols]
y = X.pop(target)

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size= 0.2, random_state= 42)

sc = StandardScaler()

x_train_sc = sc.fit_transform(x_train)
x_test_sc = sc.transform(x_test)

In [42]:
x_train_sc.shape, y_train.shape

((167, 10), (167,))

In [43]:
xgb_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

rf_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=6,
    random_state=42,
    n_jobs=-1,
)

xgb_model.fit(x_train_sc, y_train)
rf_model.fit(x_train_sc, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",6
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_

In [44]:
models = {
    'Random Forest': rf_model,
    'XGBoost': xgb_model,
}

for name, model in models.items():
    print(name)
    y_pred = model.predict(x_test_sc)
    metrics = {
        "MAE": mean_absolute_error(y_test, y_pred),
        "MSE": mean_squared_error(y_test, y_pred),
        "RMSE": root_mean_squared_error(y_test, y_pred),
        "R2": r2_score(y_test, y_pred),
        "MAPE": mean_absolute_percentage_error(y_test, y_pred),
        "MedianAE": median_absolute_error(y_test, y_pred),
        "ExplainedVariance": explained_variance_score(y_test, y_pred),
    }

    for k, v in metrics.items():
        print(f"{k:<20}: {v:.6f}")

Random Forest
MAE                 : 1.497098
MSE                 : 3.899133
RMSE                : 1.974622
R2                  : 0.862293
MAPE                : 0.081295
MedianAE            : 1.150342
ExplainedVariance   : 0.863894
XGBoost
MAE                 : 1.118312
MSE                 : 2.283656
RMSE                : 1.511177
R2                  : 0.919347
MAPE                : 0.071859
MedianAE            : 0.943055
ExplainedVariance   : 0.920547


In [36]:
del df
gc.collect()
%whos

Variable                         Type                     Data/Info
-------------------------------------------------------------------
RandomForestRegressor            ABCMeta                  <class 'sklearn.ensemble.<...>t.RandomForestRegressor'>
StandardScaler                   type                     <class 'sklearn.preproces<...>ng._data.StandardScaler'>
X                                DataFrame                Shape: (209, 10)
XGBRegressor                     type                     <class 'xgboost.sklearn.XGBRegressor'>
explained_variance_score         function                 <function explained_varia<...>_score at 0x7ff4a59f3560>
gc                               module                   <module 'gc' (built-in)>
mean_absolute_error              function                 <function mean_absolute_error at 0x7ff4a59f2ac0>
mean_absolute_percentage_error   function                 <function mean_absolute_p<...>_error at 0x7ff4a59f2d40>
mean_squared_error               function     